In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [4]:
# Load the dataset dynamically
df = pd.read_csv('train_u6lujuX_CVtuZ9i.csv')

print(df.head())

    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural           N  
2             1.0   

In [5]:
# Separate Clues (X) from Target Answers (y)
# We drop 'Loan_ID' because it has no predictive power
X = df.drop(columns=['Loan_ID', 'Loan_Status'])

# Map target string outputs ('Y', 'N') into numeric binary structure (1, 0)
y = df['Loan_Status'].map({'Y': 1, 'N': 0})

# Train/Test Split (80% training data, 20% validation testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 491 samples
Test set size: 123 samples


In [6]:
# Define Feature Groups for Custom Assembly Stations
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Credit_History', 'Property_Area']

# Build Sub-Pipelines for Preprocessing
# Station A: Numerical cleaner
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Station B: Categorical cleaner
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Stitch processing blocks together into a single ColumnTransformer layout
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

In [7]:
# Merge preprocessing and the Logistic Regression Model into a Master Pipeline
loan_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Train the entire processing assembly line on raw data
print("Training the loan application processing pipeline...")
loan_pipeline.fit(X_train, y_train)
print("Model training complete.")

Training the loan application processing pipeline...
Model training complete.


In [8]:
# Evaluate Performance
y_pred = loan_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nPipeline Test Accuracy: {accuracy * 100:.2f}%")

print("\nDetailed Performance Report:")
print(classification_report(y_test, y_pred, target_names=['Rejected (0)', 'Approved (1)']))


Pipeline Test Accuracy: 78.86%

Detailed Performance Report:
              precision    recall  f1-score   support

Rejected (0)       0.95      0.42      0.58        43
Approved (1)       0.76      0.99      0.86        80

    accuracy                           0.79       123
   macro avg       0.85      0.70      0.72       123
weighted avg       0.83      0.79      0.76       123

